# 01 Raw audit
Thin orchestration only. All logic is in `src/underwriting` and `scripts/audit_raw.py`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!rm -rf /content/influencer-underwriting
!git clone https://github.com/VAL-Jerono/influencer-underwriting.git /content/influencer-underwriting
%cd /content/influencer-underwriting
!pip install -q -e .

Cloning into '/content/influencer-underwriting'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 38 (delta 0), reused 35 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 22.42 KiB | 2.24 MiB/s, done.
/content/influencer-underwriting
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for underwriting (pyproject.toml) ... done


In [3]:
!pytest -q

....                                                                     [100%]
4 passed in 0.71s


In [4]:
# Fails clearly if the dataset path is wrong. Edit configs/paths.yaml (copy of paths.example.yaml) if needed.
!python scripts/audit_raw.py --stage --rows-limit 100000

raw_drive     = /content/drive/MyDrive/Influencer brand dataset
local_raw     = /content/influencer_data/raw
artifact_root = /content/drive/MyDrive/InfluencerUnderwriting/artifacts
staging: {'post_info.txt': 'copied', 'json_files.zip': 'copied', 'profiles_influencers.zip': 'copied', 'profiles_brands.zip': 'copied'}
wrote /content/drive/MyDrive/InfluencerUnderwriting/artifacts/reports/raw_audit.json
wrote /content/drive/MyDrive/InfluencerUnderwriting/artifacts/reports/raw_audit.md


In [5]:
# Full run (chunked). Only after the smoke run above looks right.
!python scripts/audit_raw.py --stage

raw_drive     = /content/drive/MyDrive/Influencer brand dataset
local_raw     = /content/influencer_data/raw
artifact_root = /content/drive/MyDrive/InfluencerUnderwriting/artifacts
staging: {'post_info.txt': 'cached', 'json_files.zip': 'cached', 'profiles_influencers.zip': 'cached', 'profiles_brands.zip': 'cached'}
wrote /content/drive/MyDrive/InfluencerUnderwriting/artifacts/reports/raw_audit.json
wrote /content/drive/MyDrive/InfluencerUnderwriting/artifacts/reports/raw_audit.md


In [8]:
import sys
sys.path.insert(0, '/content/influencer-underwriting/src')

from IPython.display import Markdown
from underwriting.io.paths import load_paths
p = load_paths()
Markdown((p.reports / 'raw_audit.md').read_text())

# Raw audit
Generated: 2026-09-29T13:51:30+00:00 (post rows limit: None)

## Posts
- rows: 1,601,074  | duplicate post IDs: 0
- post ID range: 0 to 1601073
- sponsored value counts: {'1': 221710, '0': 1379364}
- sponsored rate: 13.85%
- unique usernames: 38,113  | median posts per creator: 24
- image count distribution (capped at 10): {1: 1368283, 2: 125095, 3: 51609, 4: 23270, 5: 12005, 6: 6860, 7: 4192, 8: 2941, 9: 2438, 10: 4381}

## Gates
- creators with >=5 sponsored and >=5 organic: 12,002
- creators with >=3 sponsored and >=3 organic: 16,663
- creators with any sponsored post: 27,731
- JSON layouts in sample of 1000: {'7f3b6761': 838, 'a065d38f': 162}
- comment-like field non-empty share (heuristic): 0.956

## Coverage
- JSON referenced and found in archive: 1,601,074 / 1,601,074 (1.0)
- influencer profile matches (filename stem heuristic): 35,545 of 38,113 usernames
- brand profile archive members: 25,282

## Verify by eye before trusting
- raw first lines of post_info.txt: ["0\talisasia\t1\t1309041812857818435.json\t['1309041812857818435.jpg']", "1\talisasia\t0\t1311846669234786866.json\t['1311846669234786866.jpg']", "2\talisasia\t0\t1315560311952470229.json\t['1315560311952470229.jpg']"]
- influencer archive example members: ['users_influencers_SPOD/fodmap_friendly_eating', 'users_influencers_SPOD/paileaa', 'users_influencers_SPOD/a12team', 'users_influencers_SPOD/ilbulgaro89', 'users_influencers_SPOD/joililikoi']
- Confirm the image separator, the sponsored encoding, and the profile file naming against these samples.
